# Preprocessors

Preprocessors are optional. If none is configured, observations pass through unchanged. Use `@preprocess` when the records from a source need a small, explicit shape change before encoding.

This guide uses Iris data, but starts from a deliberately awkward raw shape: feature names live in a nested dictionary and the label is stored as an integer id.

In [1]:
from loguru import logger
from rich.pretty import pprint
from sklearn.datasets import load_iris

import json2vec as j2v

logger.remove()

iris = load_iris()

In [2]:
def raw_iris_record(index: int) -> dict:
    return {
        "features": dict(zip(iris.feature_names, iris.data[index])),
        "species_id": int(iris.target[index]),
    }

A transformation preprocessor returns one normalized record for each input record. It is the right choice for type coercion, renaming, flattening, or attaching derived fields.

In [3]:
@j2v.preprocess
def simplify_iris(record: dict) -> dict:
    features = record["features"]
    return {
        "sepal_length": float(features["sepal length (cm)"]),
        "petal_length": float(features["petal length (cm)"]),
        "species": str(iris.target_names[record["species_id"]]),
    }

In [4]:
pprint(simplify_iris(raw_iris_record(0)))

{'sepal_length': 5.1, 'petal_length': 1.4, 'species': 'setosa'}

A yielding preprocessor can expand one input into many outputs. That pattern is useful for windows, sessions, or any source object that should become multiple training observations.

In [5]:
@j2v.preprocess(yields=True)
def iris_records(indices):
    for index in indices:
        yield simplify_iris(raw_iris_record(index))

In [6]:
pprint(list(iris_records(range(3))))

[
│   {'sepal_length': 5.1, 'petal_length': 1.4, 'species': 'setosa'},
│   {'sepal_length': 4.9, 'petal_length': 1.4, 'species': 'setosa'},
│   {'sepal_length': 4.7, 'petal_length': 1.3, 'species': 'setosa'}
]